# Lab 3 — DATS 6103

**Name:**

This notebook mirrors the lab page. Everything here also runs in the browser at
<https://www.smajhi.com/DATS-6103/labs/numpy-I.html>, where the self-checks and the timers live; work in whichever you
prefer. The discussion half of the lab is not in this file — it happens away
from the keyboard.

**Conditions.** No AI assistants. The official Python, NumPy, Pandas,
Matplotlib and scikit-learn documentation and the course notes may be consulted
freely.

Run the self-check under each answer. It tells you whether your answer is right
without telling you what the answer is.

## Part A—Discussion

**Not submitted, and not written at the keyboard.** Work these in pairs, out loud, one of you holding the rubric card. The rubric cards are on the lab page: <https://www.smajhi.com/DATS-6103/labs/numpy-I.html>.

### A1. Say the shape out loud

*Suggested: 5 minutes.*

*Pair-programming screen, data science generalist.*

> "I have a tensor `T` of shape `(365, 25, 3)`—days, respondents, features. I am going to read you five expressions. For each one, tell me the shape of the result and, in one sentence, why."
>
> ```
> T.mean(axis=1)
> T.mean(axis=(0, 1))
> T[:, 0]
> T[:, 0:1]
> T.mean(axis=1, keepdims=True)
> ```

### A2. The line that ran and was wrong

*Suggested: 5 minutes.*

*Take-home follow-up discussion, ML-leaning role.*

> "This line is from the take-home you submitted. `X` holds observations in rows and features in columns.
>
> ```
> Z = X / X.std(axis=1)
> ```
>
> It ran without error, and your write-up says the columns are standardized. Walk me through whether that is true."

### A3. Views, copies, and the bug you cannot see

*Suggested: 5 minutes.*

*Phone screen, data-engineering-leaning role.*

> "In NumPy, when does indexing hand you a view and when does it hand you a copy? And why should I care which one I got?"

### A4. Which one would you ship?

*Suggested: 5 minutes.*

*Onsite, ML engineer.*

> "Two implementations of the same idea:
>
> ```
> def scale_a(x):
>     x = x / x.max()
>     return x
>
> def scale_b(x):
>     x /= x.max()
>     return x
> ```
>
> Which of these changes the caller's array? And which one would you put in a shared library?"

### A5. Read the traceback

*Suggested: 5 minutes.*

*Debugging screen, any data role.*

> "Three tracebacks, no code, no running anything. For each one: what did the author most likely write, and what is the fix?
>
> ```
> ValueError: operands could not be broadcast together with shapes (365,25,3) (365,3)
>
> ValueError: The truth value of an array with more than one element is ambiguous.
> Use a.any() or a.all()
>
> AxisError: axis 2 is out of bounds for array of dimension 2
> ```"

## Part B—Build

### B1. Build the study tensor

*Suggested: 8 minutes.*

The BMI study takes measurements on $25$ respondents every day for a year, recording three features per respondent: **weight** (lbs), **height** (inches), **age** (years). Fabricate it.

- `weight` is uniform on $[105, 230]$
- `height` is uniform on $[60, 75]$
- `age` is uniform on $[15, 85]$

Store it in a single array `T` laid out as **(day, respondent, feature)**, with the features in the order above. Use your own seeded generator—`np.random.default_rng(6103)`—so that your numbers are reproducible.

There is more than one way to fill this array. Whichever you choose, decide *before you type* what shape each intermediate has.

In [ ]:
rng = np.random.default_rng(6103)

T = ...

T.shape

In [ ]:
# self-check: run this after your answer above
assert isinstance(T, np.ndarray), "T should be a NumPy array"
assert T.shape == (365, 25, 3), f"expected shape (365, 25, 3), got {T.shape}"
assert T.dtype.kind == "f", f"expected a floating-point dtype, got {T.dtype}"
lo, hi = np.array([105., 60., 15.]), np.array([230., 75., 85.])
mins, maxs = T.min(axis=(0, 1)), T.max(axis=(0, 1))
assert np.all(mins >= lo), "some feature falls below its stated lower bound; check the feature order on the last axis"
assert np.all(maxs <= hi), "some feature exceeds its stated upper bound; check the feature order on the last axis"
assert np.all(mins < lo + 0.05 * (hi - lo)), "a feature does not reach near its lower bound; are you drawing from the right interval?"
assert np.all(maxs > hi - 0.05 * (hi - lo)), "a feature does not reach near its upper bound; are you drawing from the right interval?"
assert not np.allclose(T[..., 0], T[..., 1]), "two features are identical; each needs its own draw"
print("looks right")

### B2. Predict, then verify

*Suggested: 6 minutes.*

Eight expressions on `A = np.arange(24).reshape(2, 3, 4)`. **Write down all eight shapes before you run anything.** Put them in `predictions` as a list of eight tuples, in order.

```
1.  A.sum(axis=0)
2.  A.sum(axis=(0, 2))
3.  A[:, 1]
4.  A[:, 1:2]
5.  A.mean(axis=2, keepdims=True)
6.  A - A.mean(axis=2, keepdims=True)
7.  A.T
8.  A[..., -1]
```

A one-element shape is written `(3,)`, with the comma. A scalar has shape `()`.

In [ ]:
A = np.arange(24).reshape(2, 3, 4)

predictions = [
    (), # 1
    (), # 2
    (), # 3
    (), # 4
    (), # 5
    (), # 6
    (), # 7
    (), # 8
]

len(predictions)

In [ ]:
# self-check: run this after your answer above
A = np.arange(24).reshape(2, 3, 4)
actual = [
    A.sum(axis=0).shape,
    A.sum(axis=(0, 2)).shape,
    A[:, 1].shape,
    A[:, 1:2].shape,
    A.mean(axis=2, keepdims=True).shape,
    (A - A.mean(axis=2, keepdims=True)).shape,
    A.T.shape,
    A[..., -1].shape,
]
assert len(predictions) == 8, f"expected 8 predictions, got {len(predictions)}"
wrong = [i for i, (p, a) in enumerate(zip(predictions, actual), start=1) if tuple(p) != a]
assert not wrong, (
    f"wrong on item(s) {wrong}. You predicted "
    + ", ".join(f"#{i}: {tuple(predictions[i-1])}" for i in wrong)
    + ". Count the integer indices and the named axes again—do not just run the expression."
)
print("looks right")

### B3. The axis that disappears

*Suggested: 10 minutes.*

From `T`, compute each of the following. Name the axis you are collapsing before you write the call.

- `daily_feature_means`—for each day, the average of the $25$ respondents, per feature
- `person_feature_means`—for each respondent, the average over the $365$ days, per feature
- `feature_means`—the average over every day and every respondent, one number per feature
- `heaviest_per_day`—for each day, the largest weight recorded that day
- `age_spread`—the standard deviation of age over the whole study, a single number

In [ ]:
daily_feature_means = ...
person_feature_means = ...
feature_means = ...
heaviest_per_day = ...
age_spread = ...

(daily_feature_means.shape, person_feature_means.shape,
 feature_means.shape, heaviest_per_day.shape, np.shape(age_spread))

In [ ]:
# self-check: run this after your answer above
assert daily_feature_means.shape == (365, 3), f"daily_feature_means: expected (365, 3), got {daily_feature_means.shape}"
assert person_feature_means.shape == (25, 3), f"person_feature_means: expected (25, 3), got {person_feature_means.shape}"
assert feature_means.shape == (3,), f"feature_means: expected (3,), got {feature_means.shape}"
assert heaviest_per_day.shape == (365,), f"heaviest_per_day: expected (365,), got {heaviest_per_day.shape}"
assert np.shape(age_spread) == (), f"age_spread should be a single number, got shape {np.shape(age_spread)}"
# the design is balanced, so averaging the averages must return the grand mean
assert np.allclose(daily_feature_means.mean(axis=0), feature_means), \
    "averaging daily_feature_means over the days does not reproduce feature_means"
assert np.allclose(person_feature_means.mean(axis=0), feature_means), \
    "averaging person_feature_means over the respondents does not reproduce feature_means"
assert np.all(heaviest_per_day >= daily_feature_means[:, 0]), \
    "on some day the maximum weight is below that day's mean weight—check which axis you collapsed"
assert 0 < float(age_spread) < 40, "age_spread is not a plausible standard deviation for ages on [15, 85]"
print("looks right")

### B4. BMI, and centering the right axis

*Suggested: 11 minutes.*

Body mass index from imperial units is
$$
\text{BMI} = 703\cdot\frac{\text{weight in lbs}}{(\text{height in inches})^2}.
$$

1. Compute `bmi`, one value per respondent per day, shape `(365, 25)`. No loops.
2. Compute `bmi_centred`: the same array with **each day's mean subtracted from that day's readings**. Every day gets its own mean. Do it in one expression, with no loop and no `np.tile`.

The second part is the whole problem. Getting it backwards will still run.

In [ ]:
bmi = ...
bmi_centred = ...

bmi.shape, bmi_centred.shape

In [ ]:
# self-check: run this after your answer above
assert bmi.shape == (365, 25), f"bmi: expected (365, 25), got {bmi.shape}"
assert 10 < bmi.min() and bmi.max() < 60, \
    f"BMI values are implausible (min {bmi.min():.1f}, max {bmi.max():.1f})—check the formula and which feature is which"
assert bmi_centred.shape == (365, 25), f"bmi_centred: expected (365, 25), got {bmi_centred.shape}"
assert np.allclose(bmi_centred.mean(axis=1), 0, atol=1e-9), \
    "the readings within a day do not average to zero, so you have not centered each day"
assert not np.allclose(bmi_centred.mean(axis=0), 0, atol=1e-6), \
    "you centered each RESPONDENT across the year, not each DAY across the respondents—the axes are swapped"
assert np.allclose(np.diff(bmi - bmi_centred, axis=1), 0, atol=1e-9), \
    "what you subtracted is not constant within a day"
print("looks right")

### B5. A function that must not damage its input

*Suggested: 10 minutes.*

Write `cap_weights(data, limit)`. It takes a study tensor laid out like `T` and returns a **new** array in which every weight above `limit` has been replaced by `limit`. Heights and ages are untouched, and the caller's array must be exactly as it was.

In [ ]:
def cap_weights(data, limit):
    """Return a NEW array with weights capped at `limit`. `data` must not change."""
    ...

before = T.copy()
capped = cap_weights(T, 200.0)
capped.shape

In [ ]:
# self-check: run this after your answer above
assert capped is not None, "cap_weights returned None—did you forget the return?"
assert capped.shape == T.shape, f"expected shape {T.shape}, got {capped.shape}"
assert np.array_equal(T, before), "cap_weights modified the caller's array"
assert not np.shares_memory(T, capped), "the result shares memory with T—it is a view, not an independent array"
assert capped[..., 0].max() <= 200.0 + 1e-12, f"some weight is still above the cap: {capped[..., 0].max():.3f}"
assert (capped[..., 0] < 200.0).any(), "every weight is at the cap—you replaced too much"
assert np.array_equal(capped[..., 1:], T[..., 1:]), "heights or ages were altered"
assert np.array_equal(capped[..., 0][T[..., 0] <= 200.0], T[..., 0][T[..., 0] <= 200.0]), \
    "weights that were already below the cap were changed"
print("looks right")

## np.shares_memory(T, capped) is False: np.array(..., copy=True) allocates, so

## writing into `out` cannot reach T. A view would have shared the buffer.

before = T.copy()
capped = cap_weights(T, 200.0)
```

`np.minimum` caps elementwise and returns a new array; assigning it into
`out[..., 0]` writes back into the copy.

The trap is that `out = data` then `out[..., 0] = ...` would pass a shape check,
pass a cap check, and silently destroy the caller's array. `copy=True` is what
makes the contract in the docstring true, and `np.shares_memory` is how you
prove it rather than hope it.

Now check `np.shares_memory(T, capped)` yourself, and explain in a comment which line in your function is the one that broke the link.

##  B6. Rebind or mutate, deliberately

Two functions, same arithmetic, opposite contracts. *Standardize* means: subtract each column's mean, then divide by each column's standard deviation.

- `standardise_copy(x)` returns a new standardized array and leaves `x` untouched.
- `standardise_inplace(x)` modifies `x` in place and returns `None`.

Write both. The difference between them is one operator, and the self-check can tell.

```{pyodide}
#| exercise: lab3_b6
def standardise_copy(x):
    """Return a NEW standardized array. `x` must not change."""
    ...

def standardise_inplace(x):
    """Standardize `x` IN PLACE. Return None."""
    ...

data = np.random.default_rng(0).normal(loc=[5., 50., 500.], scale=[1., 10., 100.], size=(12, 3))
data.mean(axis=0).round(2)
```

```{pyodide}
#| exercise: lab3_b6
#| check: true
try:
    d0 = data.copy()
    out = standardise_copy(data)
    assert out is not None, "standardise_copy returned None"
    assert out.shape == data.shape, f"standardise_copy: expected shape {data.shape}, got {out.shape}"
    assert np.array_equal(data, d0), "standardise_copy modified its argument"
    assert np.allclose(out.mean(axis=0), 0, atol=1e-12), "standardise_copy: columns are not centered"
    assert np.allclose(out.std(axis=0), 1, atol=1e-12), "standardise_copy: columns do not have unit standard deviation"

    d1 = data.copy()
    ret = standardise_inplace(d1)
    assert ret is None, "standardise_inplace should return None"
    assert not np.array_equal(d1, data), "standardise_inplace did not change the array it was given—you rebound a local name"
    assert np.allclose(d1.mean(axis=0), 0, atol=1e-12), "standardise_inplace: columns are not centered"
    assert np.allclose(d1.std(axis=0), 1, atol=1e-12), "standardise_inplace: columns do not have unit standard deviation"
    assert np.array_equal(data, d0), "standardise_inplace reached outside its argument"
    _fb = {"correct": True, "message": """Right. The whole difference is `-=` writing into the caller's buffer where `-` allocates a new one."""}
except AssertionError as _e:
    _fb = {"correct": False,
           "message": str(_e) or "Not right yet--run it and read what came back."}
except Exception as _e:
    _fb = {"correct": False,
           "message": "{}: {}".format(type(_e).__name__, _e)}
_fb
```

```python
def standardise_copy(x):
    """Return a NEW standardized array. `x` must not change."""
    x = np.asarray(x, dtype=float)
    return (x - x.mean(axis=0)) / x.std(axis=0)

def standardise_inplace(x):
    """Standardize `x` IN PLACE. Return None."""
    m = x.mean(axis=0)
    s = x.std(axis=0)
    x -= m
    x /= s
    return None
```

Same arithmetic, and the difference is entirely in the assignment.

`standardise_copy` builds a new array with `-` and `/`, each of which allocates,
and hands it back. The argument is never written to.

`standardise_inplace` uses `-=` and `/=`, which write into the buffer the caller
owns. Writing `x = (x - m) / s` instead would rebind the local name `x` to a new
array and leave the caller's untouched—the function would appear to do nothing,
which is exactly what the check's "you rebound a local name" message is for.

Note the means are computed *before* the first write. Compute `s` after `x -= m`
and you divide by the standard deviation of the already-centered array, which is
the same number here but will not be the day you reorder the operations.

Then try `standardise_inplace(np.arange(12).reshape(4, 3))` in a scratch cell and read the error. One line of comment: why does the copying version survive that input and the in-place version not?

##  B7. The screening pass

A reading is **flagged** when its BMI exceeds $30$. Using boolean masks and no loops:

- `mask`—a boolean array, shape `(365, 25)`, true at flagged readings
- `counts`—for each day, how many of the $25$ respondents were flagged
- `worst_day`—the index of the day with the most flags
- `n_flagged`—the total number of flagged readings in the year
- `flagged_weights`—the weights of the flagged readings, as a $1$D array
- `n_flagged_and_older`—how many flagged readings also came from a respondent older than $60$

Combine conditions with `&`, not `and`, and parenthesise.

```{pyodide}
#| exercise: lab3_b7
mask = ...
counts = ...
worst_day = ...
n_flagged = ...
flagged_weights = ...
n_flagged_and_older = ...

counts.shape, int(worst_day), int(n_flagged), flagged_weights.shape
```

```{pyodide}
#| exercise: lab3_b7
#| check: true
try:
    assert mask.dtype == bool, f"mask should be boolean, got {mask.dtype}"
    assert mask.shape == (365, 25), f"mask: expected (365, 25), got {mask.shape}"
    assert np.all(bmi[mask] > 30) and np.all(bmi[~mask] <= 30), "mask does not select exactly the readings with BMI above 30"
    assert np.shape(counts) == (365,), f"counts: expected (365,), got {np.shape(counts)}"
    assert int(np.sum(counts)) == int(n_flagged), "counts does not sum to n_flagged—one of the two collapses the wrong axis"
    assert 0 <= int(worst_day) < 365, f"worst_day out of range: {worst_day}"
    assert counts[int(worst_day)] == np.max(counts), "worst_day is not the day with the largest count"
    assert np.ndim(flagged_weights) == 1, f"flagged_weights should be 1D, got {np.ndim(flagged_weights)} dimensions"
    assert flagged_weights.size == int(n_flagged), "flagged_weights has the wrong number of entries"
    assert not np.shares_memory(T, flagged_weights), "flagged_weights shares memory with T, which a boolean mask never produces—check what you indexed"
    assert 0 < int(n_flagged_and_older) < int(n_flagged), "n_flagged_and_older should be a strict subset count"
    _fb = {"correct": True, "message": """Right. A boolean array sums as ones, and boolean indexing always copies."""}
except AssertionError as _e:
    _fb = {"correct": False,
           "message": str(_e) or "Not right yet--run it and read what came back."}
except Exception as _e:
    _fb = {"correct": False,
           "message": "{}: {}".format(type(_e).__name__, _e)}
_fb
```

```python
mask                 = bmi > 30
counts               = mask.sum(axis=1)
worst_day            = counts.argmax()
n_flagged            = mask.sum()
flagged_weights      = T[:, :, 0][mask]
n_flagged_and_older  = np.sum(mask & (T[:, :, 2] > 60))
```

A boolean array sums as though `True` were 1, so `mask.sum(axis=1)` counts
flags per day and `mask.sum()` counts them everywhere. No loop and no `if`.

`T[:, :, 0][mask]` is worth pausing on. `mask` is `(365, 25)` and so is the
weight plane, so the mask selects exactly the flagged weights—and returns them
flattened, because the count differs from day to day and no rectangular shape
could hold them. Boolean indexing always copies, which is why the check asserts
`not np.shares_memory`.

`&` rather than `and`: `and` asks for one truth value from a whole array, which
is B8's second traceback.

`np.shares_memory(T, flagged_weights)` came back `False`. Write one comment saying why that was guaranteed before you ran it.

##  B8. Three tracebacks, three fixes

Each line below raises. The comment states the error and what the author *meant*. Repair each one—minimally, changing the intent of nothing—and store the results in `fix1`, `fix2`, `fix3`.

```{pyodide}
#| exercise: lab3_b8

## 1. intent: express every reading as a deviation from that day's average respondent

## ValueError: operands could not be broadcast together with shapes (365,25,3) (365,3)

## T - T.mean(axis=1)

fix1 = ...

## 2. intent: flag readings that are BOTH above BMI 30 AND from a respondent older than 60

## ValueError: The truth value of an array with more than one element is ambiguous

## (bmi > 30) and (T[:, :, 2] > 60)

fix2 = ...

## 3. intent: each respondent's average BMI over the whole year

## AxisError: axis 2 is out of bounds for array of dimension 2

## bmi.mean(axis=2)

fix3 = ...

fix1.shape, fix2.shape, fix3.shape
```

```{pyodide}
#| exercise: lab3_b8
#| check: true
try:
    assert fix1.shape == (365, 25, 3), f"fix1: expected (365, 25, 3), got {fix1.shape}"
    assert np.allclose(fix1.mean(axis=1), 0, atol=1e-9), "fix1: the 25 respondents within a day should now average to zero on every feature"
    assert np.allclose(np.diff(T - fix1, axis=1), 0, atol=1e-9), "what you subtracted is not constant within a day"

    assert fix2.dtype == bool, f"fix2 should be a boolean array, got {fix2.dtype}"
    assert fix2.shape == (365, 25), f"fix2: expected (365, 25), got {fix2.shape}"
    assert np.all(bmi[fix2] > 30), "fix2 selects a reading with BMI at or below 30"
    assert np.all(T[:, :, 2][fix2] > 60), "fix2 selects a respondent aged 60 or under"
    assert fix2.sum() < (bmi > 30).sum(), "fix2 is not stricter than the BMI condition alone—did you use | instead of &?"

    assert np.shape(fix3) == (25,), f"fix3: expected (25,), got {np.shape(fix3)}"
    assert 10 < fix3.min() and fix3.max() < 60, "fix3 does not look like a set of average BMIs"
    assert np.isclose(fix3.mean(), bmi.mean()), "fix3 averaged over the wrong axis"
    _fb = {"correct": True, "message": """Right. A broadcasting mismatch, a Python operator on an array, and an axis that does not exist -- three different faults that all read the same at first."""}
except AssertionError as _e:
    _fb = {"correct": False,
           "message": str(_e) or "Not right yet--run it and read what came back."}
except Exception as _e:
    _fb = {"correct": False,
           "message": "{}: {}".format(type(_e).__name__, _e)}
_fb
```

```python

## 1. broadcasting mismatch: (365,3) does not align with (365,25,3) from the right

fix1 = T - T.mean(axis=1, keepdims=True)

## 2. Python operator on an array: `and` wants one truth value, `&` works elementwise

fix2 = (bmi > 30) & (T[:, :, 2] > 60)

## 3. an axis that does not exist: bmi is 2-D, so the year is axis 0

fix3 = bmi.mean(axis=0)
```

Three different mistakes that all read as "numpy is being difficult".

**1 is a broadcasting mismatch.** `T.mean(axis=1)` is `(365, 3)`; broadcasting
aligns from the right, so `3` meets `3` and then `365` meets `25`. `keepdims=True`
gives `(365, 1, 3)`, which aligns. The fix does not change the intent—it keeps
the collapsed axis open so the subtraction lands where it was meant to.

**2 is a Python operator on an array.** `and` needs one `True` or `False` and an
array of 9,125 cannot supply it. `&` is the elementwise version. The parentheses
are not decoration: `&` binds tighter than `>`, so without them you get
`bmi > (30 & T[:, :, 2]) > 60`.

**3 is an axis that does not exist.** `bmi` is `(365, 25)`, so there is no axis
2; the year is axis 0. The error names the real problem for once, and the fix is
to count the axes of the array you actually have rather than the one you were
thinking of.

For each of the three, add a one-line comment naming the *category* of the mistake: a broadcasting mismatch, a Python operator used on an array, or an axis that had already been removed.

---

## Where this comes back {.unnumbered}

None of this is NumPy trivia with a shelf life of one week.

- ** (next lab).** `X - X.mean(axis=0)` is the first line of PCA. You wrote its sibling in B4 and again in B8.
- **Regression and ridge.** Design matrices are built by stacking and broadcasting, and the difference between a fitted value and a coefficient is a difference of axes.
- **The curse of dimensionality.** Pairwise distances are built with exactly the "column against row" broadcasting you used here, at a scale where a needless copy costs you the machine.
- **—clustering.** Standardizing before you cluster is B6, and doing it along the wrong axis is a mistake that will not raise.

## Submitting {.unnumbered}

**Your lab score comes from what you do in the room, not from this file.** Three
points: you took both seats in the Part A interview and your partner signed your
rubric card; you said something substantive at the checkpoint; and you were
working on Part B when I came round and could tell me where you were stuck. None
of the three rewards being right.

Hand in your partner's signed rubric card before you leave. Upload the notebook
to Blackboard too—it is the record of what you did, and I will look at it if a
grade is ever questioned, but it is not what earns the points.

## Before you leave

Upload this notebook to Blackboard. It is the record of what you did; the lab
grade itself comes from what you do in the room.

Then post a comment at the bottom of the lab page, <https://www.smajhi.com/DATS-6103/labs/numpy-I.html>. One line is enough.
It proves your GitHub account works with the comment system, which is how every
week's reading credit is earned.